### Setup

Installs everything this notebook needs — `onnx` for structural validation,
`onnxruntime` for inference, no `torch`. Safe to re-run.


In [1]:
%pip install -q onnx onnxruntime numpy


Note: you may need to restart the kernel to use updated packages.


# Inspecting and running `best_evidential.onnx` as a black box

This is a real, non-trivial model — not the toy MLP from the rest of the
workshop. We still treat it as a black box: everything below is discovered
from the `.onnx` file itself, with no knowledge of the training code that
produced it.

It turns out to have **three inputs of different rank and dtype**, and
**shared dynamic dimensions** across them (`batch` and `seq_len` both show
up in more than one input and must be kept consistent) — a step up from the
single-input MLP case in `inspect_onnx_model.ipynb`.


In [2]:
import os

import numpy as np
import onnx
import onnxruntime as ort

ONNX_PATH = "best_evidential.onnx"
assert os.path.exists(ONNX_PATH), f"{ONNX_PATH} not found in this directory"


## 1. Open the file and check it's valid


In [3]:
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

opsets = [f"{o.domain or 'ai.onnx'}={o.version}" for o in onnx_model.opset_import]
op_types = sorted(set(n.op_type for n in onnx_model.graph.node))

print(f"IR version: {onnx_model.ir_version}")
print(f"Producer:   {onnx_model.producer_name} {onnx_model.producer_version}")
print(f"Opset:      {opsets}")
print(f"Nodes:      {len(onnx_model.graph.node)}")
print(f"Op types:   {op_types}")


IR version: 10
Producer:   pytorch 2.11.0+cu130
Opset:      ['ai.onnx=18']
Nodes:      376
Op types:   ['Add', 'Cast', 'Clip', 'Concat', 'Div', 'Erf', 'Expand', 'Gather', 'Gemm', 'Greater', 'LayerNormalization', 'MatMul', 'Mul', 'Reshape', 'Shape', 'Sin', 'Slice', 'Softmax', 'Softplus', 'Squeeze', 'Transpose', 'Unsqueeze', 'Where']


The op list (`LayerNormalization`, `Softmax`, `Erf`, `MatMul`/`Gemm`,
`Transpose`) is the fingerprint of a **transformer encoder** (`Erf` is how
PyTorch's exact-GELU shows up in ONNX). We don't need to know more than that
to run it — but it's a useful sanity check that this is a sequence model,
which is why one input turns out to carry a `seq_len` axis.


## 2. Learn the input/output contract

Three inputs this time, each with its own rank, dtype, and set of dynamic
axes.


In [4]:
session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

def describe(io_list, kind):
    for meta in io_list:
        dynamic_axes = [d for d in meta.shape if isinstance(d, str)]
        print(f"{kind:<7} name={meta.name!r:<18} shape={meta.shape}  dtype={meta.type}"
              + (f"   (dynamic: {dynamic_axes})" if dynamic_axes else ""))

describe(session.get_inputs(), "input")
describe(session.get_outputs(), "output")


input   name='events'           shape=['batch', 'seq_len', 5]  dtype=tensor(float)   (dynamic: ['batch', 'seq_len'])
input   name='pad_mask'         shape=['batch', 'seq_len']  dtype=tensor(bool)   (dynamic: ['batch', 'seq_len'])
input   name='global_features'  shape=['batch', 24]  dtype=tensor(float)   (dynamic: ['batch'])
output  name='evidence'         shape=['batch', 5]  dtype=tensor(float)   (dynamic: ['batch'])


## 3. Resolve dynamic dimensions — consistently, across inputs

With a single input (the MLP case) any symbolic dim could be resolved in
isolation. Here `"batch"` appears in all three inputs and `"seq_len"` in
two of them — pick **one concrete value per symbolic name** and reuse it
everywhere, or `events` and `pad_mask` will disagree with each other and
the run will fail (or silently broadcast wrong).


In [5]:
ONNX_TO_NUMPY = {
    "tensor(float)": np.float32,
    "tensor(double)": np.float64,
    "tensor(int64)": np.int64,
    "tensor(int32)": np.int32,
    "tensor(bool)": np.bool_,
}


def concrete_shape(shape, dim_bindings):
    return tuple(dim_bindings[d] if isinstance(d, str) else d for d in shape)


def dummy_inputs(session, dim_bindings, rng):
    feed = {}
    for meta in session.get_inputs():
        shape = concrete_shape(meta.shape, dim_bindings)
        dtype = ONNX_TO_NUMPY[meta.type]
        if dtype is np.bool_:
            # Can't discover mask *semantics* (True = padding? True = valid?)
            # from the graph alone -- default to "nothing is padded" as the
            # safest guess for a blind dummy pass.
            arr = np.zeros(shape, dtype=dtype)
        else:
            arr = rng.standard_normal(size=shape).astype(dtype)
        feed[meta.name] = arr
    return feed


## 4 + 5. Build dummy inputs and run the model


In [6]:
rng = np.random.default_rng(0)
dim_bindings = {"batch": 2, "seq_len": 6}

feed = dummy_inputs(session, dim_bindings, rng)
for name, arr in feed.items():
    print(f"{name:<18} shape={arr.shape}  dtype={arr.dtype}")

outputs = session.run(None, feed)
for meta, out in zip(session.get_outputs(), outputs):
    print(f"\n{meta.name}: shape={out.shape} dtype={out.dtype}")
    print(out)


events             shape=(2, 6, 5)  dtype=float32
pad_mask           shape=(2, 6)  dtype=bool
global_features    shape=(2, 24)  dtype=float32

evidence: shape=(2, 5) dtype=float32
[[4.7723675e-01 3.8732989e+00 5.0944396e-05 4.2571798e-01 5.1438584e+00]
 [7.3603377e-02 3.0183077e-03 4.3343203e-05 2.0663120e+01 1.1772598e+00]]


`evidence` comes out as 5 numbers per sample. The name plus the transformer
fingerprint above is a strong hint this is an **evidential regression**
head (parameters like γ, ν, α, β describing a predictive distribution
rather than a single point estimate) — but that's an inference from the
name and shape, not something the graph itself states. Recovering what each
of the 5 values *means* would need the training code or documentation.


## The same code, different batch size and sequence length

`seq_len` is free to change independently of `batch` — and the output stays
`(batch, 5)` regardless, which tells us the sequence axis gets pooled away
somewhere inside the graph (consistent with a transformer that reduces over
tokens, e.g. via a CLS token or attention pooling) rather than appearing in
the output.


In [7]:
for batch, seq_len in [(1, 3), (4, 10), (2, 1)]:
    feed = dummy_inputs(session, {"batch": batch, "seq_len": seq_len}, rng)
    out = session.run(None, feed)[0]
    print(f"batch={batch:>2} seq_len={seq_len:>3}  ->  events {feed['events'].shape}  output {out.shape}")


batch= 1 seq_len=  3  ->  events (1, 3, 5)  output (1, 5)
batch= 4 seq_len= 10  ->  events (4, 10, 5)  output (4, 5)
batch= 2 seq_len=  1  ->  events (2, 1, 5)  output (2, 5)


## Takeaways

- Multi-input models need dynamic dimensions resolved **once, by name,
  and shared** across every input that mentions them — not resolved
  independently per input.
- Dtype-aware dummy generation matters: a `bool` mask filled with the same
  `standard_normal` trick used for `float` inputs would silently do the
  wrong thing (or error).
- Op-type fingerprints (`LayerNormalization`, `Softmax`, `Erf`, …) let you
  guess the *family* of a black-box model (transformer, here) even with
  zero documentation — but shapes and op types never tell you what the
  output values mean; that requires the source or docs.
